# 01c — Train FLUX.1-dev Character LoRA (ai-toolkit)  — Iteration 2

Replaces the SDXL DreamBooth approach (01b). Fixes iteration-1 problems:
- **Realism / sharp eyes+face** → FLUX.1-dev is far more photoreal than SDXL base
- **Tighter likeness** → FLUX handles identity better; rank 32
- **Steerability** → ai-toolkit trains on your PER-IMAGE JoyCaption detailed captions
  (the diffusers DreamBooth script ignored captions and used one instance prompt)
- **ComfyUI format** → ai-toolkit outputs a ComfyUI-native LoRA (fixes the zero-effect bug)

**Runtime:** A100 (40 or 80 GB). **Prereqs:**
1. Cleaned reference images + `.txt` detailed captions in
   `Drive/ai_character_studio/characters/<NAME>/reference-images/` (image + matching .txt).
   Each caption should start with the trigger token, e.g. `sks_vyuna, a woman with ...`.
2. A HuggingFace account with **FLUX.1-dev access granted** at
   https://huggingface.co/black-forest-labs/FLUX.1-dev (click 'Agree'), and a read token.

**Design notes (lessons from iter-1):**
- ai-toolkit runs in an ISOLATED uv venv (Python 3.11) so Colab's system
  numpy 2.5 / scipy / transformers can't break it (the dep hell from 01b).
- All long-running output goes to LOG FILES, never an unread PIPE (which deadlocks).

## 1. Config + mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─── Config — change these ────────────────────────────────────────────────
CHARACTER_NAME = 'Yuna'
TRIGGER_TOKEN  = 'sks_vyuna'
TRAIN_STEPS    = 2500      # FLUX: 2000-3000 typical for a person
LORA_RANK      = 32        # 16-32 typical; 32 for more identity detail
LEARNING_RATE  = 1e-4
RESOLUTIONS    = [768, 1024]   # FLUX trains well at 1024; multi-res bucketing
# ─────────────────────────────────────────────────────────────────────────

import os
DRIVE_BASE  = '/content/drive/MyDrive/ai_character_studio'
CHAR_DIR    = f'{DRIVE_BASE}/characters/{CHARACTER_NAME}'
REF_DIR     = f'{CHAR_DIR}/reference-images'
LORAS_DIR   = f'{DRIVE_BASE}/loras'
OUTPUT_LORA = f'{LORAS_DIR}/{CHARACTER_NAME}_flux.safetensors'
os.makedirs(LORAS_DIR, exist_ok=True)

# Sanity: count images and matching captions
imgs = [f for f in os.listdir(REF_DIR) if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))]
caps = [f for f in os.listdir(REF_DIR) if f.lower().endswith('.txt')]
print(f'Character: {CHARACTER_NAME} | Trigger: {TRIGGER_TOKEN}')
print(f'Images: {len(imgs)} | Caption .txt files: {len(caps)}')
missing = [f for f in imgs if not os.path.exists(os.path.join(REF_DIR, os.path.splitext(f)[0]+'.txt'))]
if missing:
    print(f'⚠️  {len(missing)} images have NO matching .txt caption:', missing[:5])
else:
    print('✅ Every image has a matching caption.')
# Peek one caption to confirm trigger is present
if caps:
    sample = open(os.path.join(REF_DIR, caps[0])).read()
    print(f'\nSample caption ({caps[0]}):\n  {sample[:200]}')
    if TRIGGER_TOKEN not in sample:
        print(f'⚠️  Trigger "{TRIGGER_TOKEN}" not found in this caption — captions should include it.')

## 2. HuggingFace login (FLUX.1-dev is gated)

In [ ]:
# FLUX.1-dev requires accepting the license + a token. Two ways:
# 1 (recommended): add HF_TOKEN to Colab Secrets (key icon, left sidebar)
# 2: paste when prompted below
import os
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab Secrets.')
except Exception:
    from getpass import getpass
    hf_token = getpass('Paste your HuggingFace token (needs FLUX.1-dev access): ')
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
# Persist HF cache to Drive so the 24GB FLUX download survives sessions
os.environ['HF_HOME'] = f'{DRIVE_BASE}/models/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

# Verify access to the gated repo
!pip install -q huggingface_hub
from huggingface_hub import HfApi
try:
    HfApi().model_info('black-forest-labs/FLUX.1-dev', token=hf_token)
    print('✅ FLUX.1-dev access confirmed.')
except Exception as e:
    print('❌ No access to FLUX.1-dev. Accept the license at')
    print('   https://huggingface.co/black-forest-labs/FLUX.1-dev  then rerun.')
    print('   Error:', str(e)[:200])

## 3. Install ai-toolkit in an isolated uv venv
Keeps it away from Colab's Python 3.13 / numpy 2.5 / scipy that broke ai-toolkit in 01b.

In [ ]:
import subprocess, os
TOOLKIT_DIR = '/content/ai-toolkit'
VENV = f'{TOOLKIT_DIR}/.venv'
PY = f'{VENV}/bin/python'

# uv = fast installer + can provision a clean Python 3.11 (isolated from Colab 3.13)
!pip install -q uv

if not os.path.exists(TOOLKIT_DIR):
    !git clone https://github.com/ostris/ai-toolkit.git {TOOLKIT_DIR}
    !cd {TOOLKIT_DIR} && git submodule update --init --recursive

# Create an isolated venv with its own Python 3.11 (uv downloads it)
!uv venv --python 3.11 {VENV}

# Install order matters:
# 1) ai-toolkit requirements
# 2) torch 2.6 LAST among torch pkgs — ai-toolkit's NVFP4 custom_op returns
#    list[torch.Tensor] (PEP585), which torch.library.infer_schema only supports on >=2.5
# 3) numpy pinned to 1.26 VERY LAST — torch/torchvision try to pull numpy 2.x, but
#    ai-toolkit's compiled C-extensions are built against numpy 1.x → 'dtype size changed'
#    ABI crash. torch 2.6 works fine with numpy 1.26.
!uv pip install --python {PY} -r {TOOLKIT_DIR}/requirements.txt
!uv pip install --python {PY} --upgrade \
    "torch==2.6.0" "torchvision==0.21.0" "torchaudio==2.6.0" \
    --index-url https://download.pytorch.org/whl/cu124
!uv pip install --python {PY} huggingface_hub hf_transfer
!uv pip install --python {PY} "numpy==1.26.4"

# Verify: torch 2.6 + numpy 1.26 + diffusers import all clean
chk = subprocess.run([PY, '-c',
    'import numpy, torch, diffusers, transformers, safetensors; '
    'from diffusers.schedulers.scheduling_dpmsolver_multistep import DPMSolverMultistepScheduler; '
    'print("numpy", numpy.__version__, "torch", torch.__version__, "cuda", torch.cuda.is_available()); '
    'print("diffusers", diffusers.__version__, "transformers", transformers.__version__)'],
    capture_output=True, text=True)
print(chk.stdout)
if chk.returncode != 0:
    print('IMPORT CHECK FAILED:'); print(chk.stderr[-800:])
else:
    print('✅ ai-toolkit venv healthy (torch 2.6 + numpy 1.26, isolated + ABI-clean).')

## 4. Build the FLUX LoRA training config (uses your detailed captions)

In [ ]:
import yaml

# On A100 80GB set quantize False (faster, best quality). On 40GB keep True.
import torch
vram_gb = torch.cuda.get_device_properties(0).total_memory/1024**3 if torch.cuda.is_available() else 0
QUANTIZE = vram_gb < 60
print(f'GPU VRAM ~{vram_gb:.0f} GB → quantize={QUANTIZE}')

config = {
  'job': 'extension',
  'config': {
    'name': f'{CHARACTER_NAME}_flux',
    'process': [{
      'type': 'sd_trainer',
      'training_folder': '/content/training_output',
      'device': 'cuda:0',
      'trigger_word': TRIGGER_TOKEN,
      'network': {'type': 'lora', 'linear': LORA_RANK, 'linear_alpha': LORA_RANK},
      'save': {'dtype': 'float16', 'save_every': 500, 'max_step_saves_to_keep': 4,
               'push_to_hub': False},
      'datasets': [{
        'folder_path': REF_DIR,
        'caption_ext': 'txt',              # ← reads your per-image detailed captions
        'caption_dropout_rate': 0.05,
        'shuffle_tokens': False,
        'cache_latents_to_disk': True,
        'resolution': RESOLUTIONS,
      }],
      'train': {
        'batch_size': 1,
        'steps': TRAIN_STEPS,
        'gradient_accumulation_steps': 1,
        'train_unet': True,
        'train_text_encoder': False,       # FLUX: text encoders stay frozen
        'gradient_checkpointing': True,
        'noise_scheduler': 'flowmatch',    # FLUX uses flow matching
        'optimizer': 'adamw8bit',
        'lr': LEARNING_RATE,
        'dtype': 'bf16',
      },
      'model': {
        'name_or_path': 'black-forest-labs/FLUX.1-dev',
        'is_flux': True,
        'quantize': QUANTIZE,              # 8-bit to fit 40GB; off on 80GB
      },
      'sample': {
        'sampler': 'flowmatch',
        'sample_every': 250,
        'width': 1024, 'height': 1024,
        'prompts': [
          f'{TRIGGER_TOKEN}, portrait photo, detailed face, sharp eyes, natural skin texture, soft window light',
          f'{TRIGGER_TOKEN}, full body, standing on a city street, candid, golden hour',
          f'{TRIGGER_TOKEN}, close-up, smiling, cinematic lighting, shallow depth of field',
        ],
        'neg': '',                          # FLUX ignores negative prompts
        'seed': 42, 'walk_seed': True,
        'guidance_scale': 4,                # FLUX distilled guidance
        'sample_steps': 20,
      },
    }],
    'meta': {'name': '[name]', 'version': '1.0'},
  }
}

CONFIG_PATH = f'/content/{CHARACTER_NAME}_flux_config.yaml'
with open(CONFIG_PATH, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)
print('Config written:', CONFIG_PATH)
print(f'  base=FLUX.1-dev | steps={TRAIN_STEPS} | rank={LORA_RANK} | res={RESOLUTIONS} | quantize={QUANTIZE}')

## 5. Train (runs in the isolated venv; logs to file, not PIPE)

In [ ]:
import subprocess, time, os, threading
os.makedirs('/content/training_output', exist_ok=True)
LOG = '/content/flux_train.log'

# Pass HF token + cache through to the venv process
env = {**os.environ, 'HF_HUB_ENABLE_HF_TRANSFER': '1'}

print('Starting FLUX LoRA training... first run downloads FLUX.1-dev (~24GB) to Drive HF cache.')
print(f'Live log: {LOG}\n')
start = time.time()
with open(LOG, 'w') as logf:
    proc = subprocess.Popen([PY, f'{TOOLKIT_DIR}/run.py', CONFIG_PATH],
                            cwd=TOOLKIT_DIR, stdout=logf, stderr=subprocess.STDOUT, env=env)

# Tail the log live in the cell while training runs (drains nothing — reads the file)
last = 0
while proc.poll() is None:
    time.sleep(15)
    txt = open(LOG).read()
    if len(txt) > last:
        # print only new tail lines to keep output manageable
        new = txt[last:]
        tail = '\n'.join(new.splitlines()[-4:])
        print(tail)
        last = len(txt)

print(f'\nTraining process exited ({proc.returncode}) in {(time.time()-start)/60:.1f} min')
print('Last log lines:')
print(subprocess.run(['tail','-n','20',LOG], capture_output=True, text=True).stdout)

## 6. Copy the trained LoRA to Drive

In [ ]:
import glob, shutil, os
cands = glob.glob(f'/content/training_output/{CHARACTER_NAME}_flux/*.safetensors')
if not cands:
    cands = glob.glob('/content/training_output/**/*.safetensors', recursive=True)
# prefer the final (highest step / latest mtime), skip optimizer files
cands = [c for c in cands if 'optimizer' not in c.lower()]
if cands:
    latest = max(cands, key=os.path.getmtime)
    shutil.copyfile(latest, OUTPUT_LORA)
    # also stage into ComfyUI loras if that dir exists this session
    comfy_loras = '/content/ComfyUI/models/loras'
    if os.path.isdir(comfy_loras):
        shutil.copyfile(latest, f'{comfy_loras}/{os.path.basename(OUTPUT_LORA)}')
    print(f'✅ LoRA saved: {OUTPUT_LORA}')
    print(f'   source: {latest}  ({os.path.getsize(OUTPUT_LORA)/1024**2:.1f} MB)')
else:
    print('ERROR: no .safetensors in training_output. Check the log above.')
    print(subprocess.run(['ls','-R','/content/training_output'], capture_output=True, text=True).stdout[:1000])

## 7. Update character metadata

In [ ]:
import json, os
meta_path = f'{CHAR_DIR}/metadata.json'
meta = {}
if os.path.exists(meta_path):
    meta = json.load(open(meta_path))
meta.update({
    'name': CHARACTER_NAME,
    'trigger': TRIGGER_TOKEN,
    'base_model': 'flux-dev',
    'flux_lora_path': OUTPUT_LORA,
    'train_steps': TRAIN_STEPS,
    'lora_rank': LORA_RANK,
    'trainer': 'ai-toolkit',
})
json.dump(meta, open(meta_path,'w'), indent=2)
print('metadata.json updated:'); print(json.dumps(meta, indent=2))
print('\n✅ Done. Validate in ComfyUI with the FLUX stills workflow + ADetailer face pass.')
print('   FLUX inference needs: flux1-dev UNet, ae.safetensors VAE, t5xxl + clip_l text encoders (see 00 setup).')